In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ---- 1. System deps ----
!pip install -q pyyaml==6.0.1

# ---- 2. PyTorch (Colab usually has a compatible version pre-installed) ----
import torch
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())

# ---- 3. Detectron2 (build from source to match installed torch/CUDA) ----
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

# ---- 4. OpenCV is already on Colab, but ensure it's present ----
!pip install -q opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires PyYAML<7.0.0,>=6.0.2, but you have pyyaml 6.0.1 which is incompatible.
Torch: 2.10.0+cu128 CUDA: True
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 20.6 MB/s eta 0:00:00


In [15]:
import cv2
import torch
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.data import MetadataCatalog, DatasetCatalog
import shutil
# ---- CONFIG ----
INPUT_VIDEO   = "fog1.mp4"
OUTPUT_VIDEO  = "rcnn_output_annotated.mp4"
MODEL_WEIGHTS = "/content/drive/My Drive/final2/model_final.pth"
SCORE_THRESH  = 0.5

# Must match the order used when you registered dawn_train/val/test
CLASSES = ["truck", "person", "bicycle", "car", "motorcycle", "bus"]

# # Register metadata so the Visualizer shows class names + colors
# if "dawn_infer" not in MetadataCatalog.list():
#     MetadataCatalog.get("dawn_infer").set(thing_classes=CLASSES)
# metadata = MetadataCatalog.get("dawn_infer")

# ---- FORCE-CLEAN metadata (removes stale 7-class cache) ----
INFER_DS = "dawn_video_infer"
if INFER_DS in DatasetCatalog:
    DatasetCatalog.remove(INFER_DS)
if INFER_DS in MetadataCatalog:
    MetadataCatalog.remove(INFER_DS)
MetadataCatalog.get(INFER_DS).set(thing_classes=CLASSES)
metadata = MetadataCatalog.get(INFER_DS)

# Verify
print("thing_classes:", metadata.thing_classes)
assert len(metadata.thing_classes) == 6, "Expected 6 classes!"

# ---- BUILD CFG (matches training) ----
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(CLASSES)
cfg.MODEL.MASK_ON = False                     # DAWN is bbox-only
cfg.MODEL.WEIGHTS = MODEL_WEIGHTS
# cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST   = 0.5
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

predictor = DefaultPredictor(cfg)

# ---- Sanity check: confirm model head shape ----
num_out = cfg.MODEL.ROI_HEADS.NUM_CLASSES
print(f"Model NUM_CLASSES = {num_out}")

# ---- VIDEO I/O ----
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Could not open {INPUT_VIDEO}")

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS) or 25
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

# ---- INFERENCE LOOP ----
frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break

    outputs = predictor(frame)                      # BGR numpy in
    instances = outputs["instances"].to("cpu")

    vis = Visualizer(
        frame[:, :, ::-1],                          # BGR -> RGB
        metadata=metadata,
        scale=1.0,
        instance_mode=ColorMode.IMAGE,
    )
    vis_out = vis.draw_instance_predictions(instances)
    annotated = vis_out.get_image()[:, :, ::-1]     # RGB -> BGR

    writer.write(annotated)
    frame_idx += 1
    if frame_idx % 30 == 0:
        print(f"Processed {frame_idx}/{total} frames")

cap.release()
writer.release()
print(f"Done. Saved to {OUTPUT_VIDEO}")

thing_classes: ['truck', 'person', 'bicycle', 'car', 'motorcycle', 'bus']
Model NUM_CLASSES = 6
Processed 30/195 frames
Processed 60/195 frames
Processed 90/195 frames
Processed 120/195 frames
Processed 150/195 frames
Processed 180/195 frames
Done. Saved to rcnn_output_annotated.mp4


In [11]:
from pathlib import Path
import shutil

folder_name = "final2"
destination_dir = Path('/content/drive/My Drive') / folder_name
destination_dir.mkdir(parents=True, exist_ok=True) # Create the directory if it doesn't exist
shutil.copy('/content/rcnn_output_annotated.mp4', destination_dir)

'/content/drive/My Drive/final2/rcnn_output_annotated.mp4'